# 🦙 Meta-Llama-3-8B Fine-Tuning on Google Colab (Free T4 GPU)
### Task: High-Performance Romanized Hinglish-to-English Neural Machine Translation

This notebook enables you to fine-tune **Meta's flagship Llama-3 (8 Billion parameters)** on the custom web-scraped social media corpus using **4-bit QLoRA** (`bitsandbytes` NF4).

#### Why Meta-Llama-3-8B?
- **8 Billion Parameters**: Tremendous reasoning and cross-lingual capacity compared to 2B and 300M models.
- **128k Tokenizer Vocabulary**: Superior byte-level and subword compression for Romanized Indic words.
- **Designed to Outperform Baselines**: Tailored to beat published baselines (RCMT @ 14.00 BLEU, PACMANtrans @ 18.66 BLEU, and Google mT5 @ 19.94 BLEU).

#### Target Runtime:
- **Google Colab Free T4 GPU** (~15.3 GB VRAM)
- **VRAM Footprint**: ~7.5 to 8.5 GB (Zero OOM risk via Paged AdamW 8-bit & gradient checkpointing)
- **Training Time**: ~45–55 minutes for 3 epochs

## Step 1: Verify Free T4 GPU Runtime
Ensure your Colab runtime is set to **GPU** (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi

## Step 2: Clone Repository & Enter Directory

In [ ]:
# Clone the repository
!git clone https://github.com/Nickhasntlost/NLP.git
%cd NLP

## Step 3: Install Required Dependencies
Installs PyTorch, Hugging Face Transformers, PEFT, BitsAndBytes for 4-bit QLoRA, and the evaluation metric suites.

In [ ]:
!pip install -q -U torch transformers peft accelerate bitsandbytes sacrebleu evaluate rouge_score bert_score

## Step 4: Hugging Face Authentication (Required for Meta-Llama-3)
Meta-Llama-3-8B is a gated model. You must:
1. Request access on Hugging Face: [https://huggingface.co/meta-llama/Meta-Llama-3-8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B)
2. Copy your User Access Token from: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Enter your token below or add it as a secret named `HF_TOKEN` in Colab's left sidebar (🔑 icon).

In [ ]:
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Authenticated using Colab HF_TOKEN secret!")
except Exception:
    from huggingface_hub import login
    login()
    print("Hugging Face login complete!")

## Step 5: Run 4-Bit QLoRA Fine-Tuning
Fine-tunes `meta-llama/Meta-Llama-3-8B` using:
- 4-bit NormalFloat (NF4) quantization
- LoRA rank $r=32$, $\alpha=64$ across all 7 linear projection layers
- Strict loss masking (calculates loss only on English translation)
- 3 Epochs on ~9,769 curated Hinglish-English sentence pairs

In [ ]:
!python finetune/finetune_llama3.py \
    --base_model meta-llama/Meta-Llama-3-8B \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 2e-4 \
    --lora_r 32 \
    --max_length 192

## Step 6: Multi-Metric Benchmark Evaluation
Evaluates the newly trained Llama-3 adapter on the 150-sample blind validation set and compares against published benchmarks (RCMT LREC-COLING 2024, PACMANtrans ICON 2023).

In [ ]:
!python finetune/evaluate_colab.py --model llama3 --limit 150

## Step 7: Interactive Translation Test
Test your fine-tuned Llama-3 model on custom sentences.

In [ ]:
test_sentences = [
    "bhaaaai kyaaa scene hai kal ka? plzz batao",
    "Aaj ka din bht zyada thk gya hu bhai",
    "Sir aap bahut achha padhate ho, mujhe samajh aa gaya.",
    "koi nhi bolega scripted h ye video",
    "Maine 636 tiktok videos khud report kiye the",
    "Yaar main rice khate hue dekh rahi thi aur mera mood kharab ho gaya."
]

for s in test_sentences:
    print(f"\nInput  : {s}")
    !python finetune/translate_llama3.py --text "{s}"

## Step 8: Package LoRA Adapter & Download
Zips the trained adapter weights (~120 MB) so you can download them directly to your laptop or upload them to Hugging Face.

In [ ]:
import shutil
from google.colab import files

# Create zip bundle
shutil.make_archive('llama3_hinglish_lora', 'zip', 'finetune/outputs/llama3_hinglish_lora')
print("LoRA adapter zipped as: llama3_hinglish_lora.zip")

# Download to local machine
files.download('llama3_hinglish_lora.zip')